In [ ]:
!pip install -U "transformers" "huggingface_hub" "tokenizers"


In [ ]:
import json
import pandas as pd
import numpy as np
import json
import pandas as pd
import numpy as np
import os, math, pandas as pd, numpy as np
from dataclasses import dataclass
from typing import Dict, List, Union, Optional
import torch
from typing import List, Sequence, Tuple
from transformers import (AutoTokenizer, AutoConfig, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from sklearn.metrics import accuracy_score, f1_score
from scipy.stats import spearmanr, pearsonr
from peft import LoraConfig, get_peft_model, TaskType
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel, AutoConfig
from tqdm import tqdm
import os
import random
import torch
from transformers import AutoTokenizer, AutoModel, AutoConfig
from peft import PeftModel
from collections import deque
from tqdm import tqdm
import os
import torch
import torch.optim as optim
import numpy as np
from transformers import get_scheduler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from torch.cuda.amp import autocast, GradScaler
import kagglehub
import random
from scipy.stats import spearmanr

from types import SimpleNamespace

In [ ]:
TASK_TYPE = "pair_cls"
MAX_LENGTH = 256
BATCH_SIZE = 16
epochs = 8
LR = 2e-5
warmup_ratio = 0.06
save_dir = 'ckpt1'
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

###
blending_model_special_token = "_" # teacher
base_model_special_token = "##" # student

In [ ]:
def info_nce(q, k, temperature=0.07, neg_valid_mask=None):
    q = F.normalize(q, dim=-1)
    k = F.normalize(k, dim=-1)

    logits = torch.matmul(q, k.T) / temperature
    labels = torch.arange(q.size(0), device=q.device)
    loss_inbatch = F.cross_entropy(logits, labels) 
    return loss_inbatch, logits

In [ ]:


class TextPairRaw(Dataset):
    def __init__(self, df: pd.DataFrame, task: str):
        self.task = task
        if task == "single_cls":
            self.samples = [(t, None, int(y)) for t, y in zip(df["text"].astype(str), df["label"].astype(int))]
        elif task == "pair_cls":
            self.samples = [(a, b) for a,b in zip(df["premise"].astype(str),
                                                            df["hypothesis"].astype(str))]
        else:  # pair_reg
            self.samples = [(a, b) for a,b in zip(df["sentence1"].astype(str),
                                                              df["sentence2"].astype(str))]
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx] 
from typing import List, Tuple, Optional



class DualTokenizerCollate:
    def __init__(self, tok_student, tok_teacher, task: str, max_len: int):
        self.ts = tok_student
        self.tt = tok_teacher
        self.task = task
        self.max_len = max_len

    def __call__(self, batch: List[Tuple[str, Optional[str], float]]):
        s1s, s2s = zip(*batch)

        if self.task == "single_cls":
            s_enc = self.ts(list(s1s), max_length=self.max_len, truncation=True,
                            padding=True, return_tensors="pt",
                            return_special_tokens_mask=True)
            t_enc = self.tt(list(s1s), max_length=self.max_len, truncation=True,
                            padding=True, return_tensors="pt",
                            return_special_tokens_mask=True)

            out = {
                "input_ids_stu": s_enc["input_ids"],
                "attention_mask_stu": s_enc["attention_mask"],
                "special_tokens_mask_stu": s_enc["special_tokens_mask"],
                "input_ids_tea": t_enc["input_ids"],
                "attention_mask_tea": t_enc["attention_mask"],
                "special_tokens_mask_tea": t_enc["special_tokens_mask"],
                "labels": torch.tensor(ys, dtype=torch.long),
            }
            if "token_type_ids" in s_enc:
                out["token_type_ids_stu"] = s_enc["token_type_ids"]
            if "token_type_ids" in t_enc:
                out["token_type_ids_tea"] = t_enc["token_type_ids"]
            return out

        # ------- pair (bi-encoder) -------
        s1_enc = self.ts(list(s1s), max_length=self.max_len, truncation=True,
                         padding=True, return_tensors="pt",
                         return_special_tokens_mask=True)
        s2_enc = self.ts(list(s2s), max_length=self.max_len, truncation=True,
                         padding=True, return_tensors="pt",
                         return_special_tokens_mask=True)

        t1_enc = self.tt(list(s1s), max_length=self.max_len, truncation=True,
                         padding=True, return_tensors="pt",
                         return_special_tokens_mask=True)
        t2_enc = self.tt(list(s2s), max_length=self.max_len, truncation=True,
                         padding=True, return_tensors="pt",
                         return_special_tokens_mask=True)

        out = {
            # student
            "input_ids1_stu": s1_enc["input_ids"],
            "attention_mask1_stu": s1_enc["attention_mask"],
            "special_tokens_mask1_stu": s1_enc["special_tokens_mask"],
            "input_ids2_stu": s2_enc["input_ids"],
            "attention_mask2_stu": s2_enc["attention_mask"],
            "special_tokens_mask2_stu": s2_enc["special_tokens_mask"],
            # teacher
            "input_ids1_tea": t1_enc["input_ids"],
            "attention_mask1_tea": t1_enc["attention_mask"],
            "special_tokens_mask1_tea": t1_enc["special_tokens_mask"],
            "input_ids2_tea": t2_enc["input_ids"],
            "attention_mask2_tea": t2_enc["attention_mask"],
            "special_tokens_mask2_tea": t2_enc["special_tokens_mask"],
        }
        # chỉ thêm token_type_ids nếu tồn tại
        if "token_type_ids" in s1_enc:
            out["token_type_ids1_stu"] = s1_enc["token_type_ids"]
        if "token_type_ids" in s2_enc:
            out["token_type_ids2_stu"] = s2_enc["token_type_ids"]
        if "token_type_ids" in t1_enc:
            out["token_type_ids1_tea"] = t1_enc["token_type_ids"]
        if "token_type_ids" in t2_enc:
            out["token_type_ids2_tea"] = t2_enc["token_type_ids"]

        return out



# def smart_read(base, name_csv, name_tsv):
#     p_csv = os.path.join(base, name_csv)
#     p_tsv = os.path.join(base, name_tsv)
#     if os.path.exists(p_csv):
#         return pd.read_csv(p_csv)
#     if os.path.exists(p_tsv):
#         return pd.read_csv(p_tsv, sep="\t")
#     raise FileNotFoundError(f"Không thấy {p_csv} hoặc {p_tsv}")


# TASK_TYPE = "pair_cls"
# train_df = smart_read(BASE_INPUT, "train.csv", "train.tsv")
# dev_df   = smart_read(BASE_INPUT, "dev.csv",   "dev.tsv")
# test_df  = smart_read(BASE_INPUT, "test.csv",  "test.tsv")
# assert {"premise","hypothesis","label"}.issubset(train_df.columns), "SciTail cần premise,hypothesis"
print("Done Prepare Dataset")

In [ ]:
data_train = '/kaggle/input/final-data/final_data.csv'
df = pd.read_csv(data_train)
df.head(3)

In [ ]:
df["premise"] = df["text"]
df["hypothesis"] = df["text"]
cols = ["premise", "hypothesis"]  # hoặc ["premise","hypothesis","label"] nếu cần placeholder
df_out = df[cols].copy()

In [ ]:
tok_student = AutoTokenizer.from_pretrained("Qwen/Qwen3-Embedding-0.6B")
model_student = AutoModel.from_pretrained(
    "Qwen/Qwen3-Embedding-0.6B",
    output_hidden_states=True
)
tok_teacher = AutoTokenizer.from_pretrained("BAAI/bge-m3")
model_teacher = AutoModel.from_pretrained(
    "BAAI/bge-m3",
    output_hidden_states=True
)

In [ ]:

train_ds = TextPairRaw(df_out, TASK_TYPE)

collate = DualTokenizerCollate(tok_student, tok_teacher, TASK_TYPE, MAX_LENGTH)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate, pin_memory=True, num_workers=2, persistent_workers=True)
print("Done Prepare Dataloader")

In [ ]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from tqdm import tqdm
import torch.nn.functional as F
from scipy.stats import pearsonr, spearmanr

tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-Embedding-0.6B')

class STSDataset(Dataset):
    def __init__(self, file_path):
        self.dataset = pd.read_csv(file_path)

    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        # instruction = "Given a text, Retrieve semantically similar text: "
        instruction=""
        return {
            "sentence1": instruction + self.dataset.iloc[idx]['sentence1'],
            "sentence2": instruction + self.dataset.iloc[idx]['sentence2'],
            "label": torch.tensor(self.dataset.iloc[idx]['score'], dtype=torch.float),
        }
        
def collate_fn(batch, tokenizer, max_len=128):
    s1_list = [item["sentence1"] for item in batch]
    s2_list = [item["sentence2"] for item in batch]
    labels = torch.stack([item["label"] for item in batch])

    enc1 = tokenizer(
        s1_list,
        truncation=True,
        padding=True,       # chỉ pad theo câu dài nhất trong batch
        max_length=max_len,
        return_tensors="pt"
    )
    enc2 = tokenizer(
        s2_list,
        truncation=True,
        padding=True,
        max_length=max_len,
        return_tensors="pt"
    )

    return {
        "input_ids1": enc1["input_ids"],
        "attention_mask1": enc1["attention_mask"],
        "input_ids2": enc2["input_ids"],
        "attention_mask2": enc2["attention_mask"],
        "labels": labels,
    }


# Modified eval_sts function with Matryoshka dimensions
def eval_sts(model, eval_loader, matryoshka_dims=[16, 32, 64, 128, 256, 512, 1024]):
    results = {}
    device = model.device

    for dim in matryoshka_dims:
        preds, labels = [], []
        
        with torch.cuda.amp.autocast(dtype=torch.float16):
            with torch.no_grad():
                for batch in tqdm(eval_loader, desc=f"Eval dim={dim}"):
                    input_ids1 = batch["input_ids1"].to(device)
                    attn1 = batch["attention_mask1"].to(device)
                    input_ids2 = batch["input_ids2"].to(device)
                    attn2 = batch["attention_mask2"].to(device)
                    label = batch["labels"]

                    out1 = model(input_ids=input_ids1, attention_mask=attn1)
                    out2 = model(input_ids=input_ids2, attention_mask=attn2)


                    emb1 = out1.last_hidden_state[:, 0, :][:, :dim]
                    emb2 = out2.last_hidden_state[:, 0, :][:, :dim]
                    # cosine similarity 
                    sim = F.cosine_similarity(emb1, emb2)
                    score = (sim + 1) * 2.5  # scale [-1,1] -> [0,5]

                    preds.extend(score.cpu().numpy())
                    labels.extend(label.numpy())

        spearman_corr, _ = spearmanr(preds, labels)
        results[f"dim_{dim}"] = spearman_corr
        print(f"  Dim {dim}: Spearman = {spearman_corr:.4f}")

    return results


# Modified eval_sts_task
def eval_sts_task(model, path_list):
    model.eval()
    print(' ✅ eval_sts_task')
    for path in path_list:
        print(f"\n{path}")
        eval_dataset = STSDataset(path)
        eval_loader = DataLoader(
            eval_dataset,
            batch_size=64,
            shuffle=False,
            collate_fn=lambda x: collate_fn(x, tokenizer)
        )
        results = eval_sts(model, eval_loader)
    model.train()

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
import datasets
import numpy as np
import torch

# Modified eval_cls function with Matryoshka dimensions
def eval_cls(model, eval_loader, dim=1024):
    preds, labels = [], []
    device = model.device

    with torch.cuda.amp.autocast(dtype=torch.float16):
        with torch.no_grad():
            for batch in tqdm(eval_loader, desc=f"Extract dim={dim}"):
                input_ids1 = batch["input_ids1"].to(device)
                attn1 = batch["attention_mask1"].to(device)
                label = batch["labels"]

                out1 = model(input_ids=input_ids1, attention_mask=attn1)
                emb1 = out1.last_hidden_state[:, 0, :][:, :dim]

                preds.extend(emb1.cpu().numpy())
                labels.extend(label.numpy())

    return preds, labels





class ClasssifyDataset(Dataset):
    def __init__(self, file_path):
        self.dataset = pd.read_csv(file_path)

    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        return {
            "text": self.dataset.iloc[idx]['text'],
            "label": torch.tensor(self.dataset.iloc[idx]['label'], dtype=torch.long),
        }

def clf_collate_fn(batch, tokenizer, max_len=512):
    s1_list = [item["text"] for item in batch]
    labels = torch.stack([item["label"] for item in batch])

    enc1 = tokenizer(
        s1_list,
        truncation=True,
        padding=True,       # chỉ pad theo câu dài nhất trong batch
        max_length=max_len,
        return_tensors="pt"
    )

    return {
        "input_ids1": enc1["input_ids"],
        "attention_mask1": enc1["attention_mask"],
        "labels": labels,
    }


# Modified eval_classification_task
def eval_classification_task(model, path_list):
    model.eval()
    print(' ✅ eval classifier')
    matryoshka_dims = [16, 32, 64, 128, 256, 512, 1024]

    for train_path, dev_path in path_list:
        print(f"\n{dev_path}")
        
        for dim in matryoshka_dims:
            eval_dataset = ClasssifyDataset(dev_path)
            eval_loader = DataLoader(
                eval_dataset,
                batch_size=64,
                shuffle=False,
                collate_fn=lambda x: clf_collate_fn(x, tokenizer)
            )

            train_dataset = ClasssifyDataset(train_path)
            train_loader = DataLoader(
                train_dataset,
                batch_size=64,
                shuffle=False,
                collate_fn=lambda x: clf_collate_fn(x, tokenizer)
            )

            X_train, y_train = eval_cls(model, train_loader, dim=dim)
            X_test, y_test = eval_cls(model, eval_loader, dim=dim)

            clf = LogisticRegression(
                random_state=42,
                n_jobs=1,
                max_iter=200,
                verbose=0,
            )
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)

            accuracy = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred, average="macro")
            print(f"  Dim {dim}: Accuracy = {accuracy:.4f}, F1 = {f1:.4f}")

    model.train()
        


In [ ]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from tqdm import tqdm
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, average_precision_score

# Qwen/Qwen3-Embedding-0.6B
# google-bert/bert-base-uncased
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-Embedding-0.6B')

class PairDataset(Dataset):
    def __init__(self, file_path):
        self.dataset = pd.read_csv(file_path)

    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        # instruction = "Given a text, Retrieve semantically similar text: "
        instruction=""
        return {
            "sentence1": instruction + self.dataset.iloc[idx]['sentence1'],
            "sentence2": instruction + self.dataset.iloc[idx]['sentence2'],
            "label": torch.tensor(self.dataset.iloc[idx]['label'], dtype=torch.float),
        }
        

# Modified eval_pair function with Matryoshka dimensions
def eval_pair(model, eval_loader, matryoshka_dims=[16, 32, 64, 128, 256, 512, 1024]):
    results = {}
    device = model.device

    for dim in matryoshka_dims:
        preds, labels = [], []
        
        with torch.cuda.amp.autocast(dtype=torch.float16):
            with torch.no_grad():
                for batch in tqdm(eval_loader, desc=f"Eval dim={dim}"):
                    input_ids1 = batch["input_ids1"].to(device)
                    attn1 = batch["attention_mask1"].to(device)
                    input_ids2 = batch["input_ids2"].to(device)
                    attn2 = batch["attention_mask2"].to(device)
                    label = batch["labels"]

                    out1 = model(input_ids=input_ids1, attention_mask=attn1)
                    out2 = model(input_ids=input_ids2, attention_mask=attn2)

                    emb1 = out1.last_hidden_state[:, 0, :][:, :dim]
                    emb2 = out2.last_hidden_state[:, 0, :][:, :dim]

                    # cosine similarity
                    sim = F.cosine_similarity(emb1, emb2)
                    score = (sim + 1) / 2

                    preds.extend(score.cpu().numpy())
                    labels.extend(label.numpy())

        metric = get_metric_pair_classification(preds, labels)
        results[f"dim_{dim}"] = metric
        print(f"  Dim {dim}: {metric}")

    return results
    
    metric = get_metric_pair_classification(preds, labels)
    print(metric)

    return metric

def get_metric_pair_classification(scores, labels):
    best_acc, best_thr = 0, 0
    for thr in np.linspace(0, 1, 200):
        preds = (scores >= thr).astype(int)
        acc = accuracy_score(labels, preds)
        if acc > best_acc:
            best_acc, best_thr = acc, thr
    preds = (scores >= best_thr).astype(int)
    return {
        "best_threshold": best_thr,
        "accuracy": best_acc,
        "f1": f1_score(labels, preds, average="macro"),
        "precision": precision_score(labels, preds, average="macro"),
        "recall": recall_score(labels, preds, average="macro"),
        "average_precision": average_precision_score(labels, scores)
    }


# Modified eval_pair_task
def eval_pair_task(model, path_list):
    model.eval()
    print(' ✅ eval_pair_task')
    for path in path_list:
        print(f"\n{path}")
        eval_dataset = PairDataset(path)
        eval_loader = DataLoader(
            eval_dataset,
            batch_size=64,
            shuffle=False,
            collate_fn=lambda x: collate_fn(x, tokenizer)
        )
        results = eval_pair(model, eval_loader)
    model.train()

In [ ]:
eval_cls_tasks = [('/kaggle/input/multitask-data/multi-data/banking_train.csv', 
                   '/kaggle/input/multitask-data/multi-data/banking77_validation.csv'),
                  ('/kaggle/input/multitask-data/multi-data/emotion_train.csv', 
                   '/kaggle/input/multitask-data/multi-data/emotion_validation.csv'), 
                  ('/kaggle/input/multitask-data/multi-data/tweet_train.csv', 
                   '/kaggle/input/multitask-data/multi-data/tweet_validation.csv')]

eval_sts_tasks = ['/kaggle/input/multitask-data/multi-data/sick_validation.csv', 
                  '/kaggle/input/multitask-data/multi-data/sts12_validation.csv', 
                  '/kaggle/input/multitask-data/multi-data/stsb_validation.csv']

eval_pair_tasks = ['/kaggle/input/multitask-data/multi-data/mrpc_validation.csv', 
                   '/kaggle/input/multitask-data/multi-data/scitail_validation.csv', 
                   '/kaggle/input/multitask-data/multi-data/wic_validation.csv']
test_cls_tasks = [('/kaggle/input/multitask-data/multi-data/banking_train.csv', 
                   '/kaggle/input/multitask-data/multi-data/banking77_test.csv'),
                  ('/kaggle/input/multitask-data/multi-data/emotion_train.csv', 
                   '/kaggle/input/multitask-data/multi-data/emotion_test.csv'), 
                  ('/kaggle/input/multitask-data/multi-data/tweet_train.csv', 
                   '/kaggle/input/multitask-data/multi-data/tweet_test.csv')]

test_sts_tasks = ['/kaggle/input/multitask-data/multi-data/sick_test.csv', 
                  '/kaggle/input/multitask-data/multi-data/sts12_test.csv', 
                  '/kaggle/input/multitask-data/multi-data/stsb_test.csv',
                 '/kaggle/input/sts-data/more_test_data/sick_r.csv', 
                  '/kaggle/input/sts-data/more_test_data/sts13.csv', 
                  '/kaggle/input/sts-data/more_test_data/sts14.csv',
                  '/kaggle/input/sts-data/more_test_data/sts15.csv',
                  '/kaggle/input/sts-data/more_test_data/sts16.csv']

test_pair_tasks = ['/kaggle/input/multitask-data/multi-data/mrpc_test.csv', 
                   '/kaggle/input/multitask-data/multi-data/scitail_test.csv', 
                   '/kaggle/input/multitask-data/multi-data/wic_test.csv']






In [ ]:
best_f1_macro = -1.0
all_preds, all_labels = [], []
total_loss = 0.0

scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


num_steps = len(train_loader)
total_traning_steps = num_steps * epochs
warmup_ratio = 0.1

def pick_devices():
    if torch.cuda.device_count() >= 2:
        dev_s = torch.device("cuda:0")  # student
        dev_t = torch.device("cuda:1")  # teacher
    elif torch.cuda.is_available():
        print("[WARN] Only 1 GPU available -> both on cuda:0")
        dev_s = dev_t = torch.device("cuda:0")
    else:
        print("[WARN] No GPU -> CPU")
        dev_s = dev_t = torch.device("cpu")
    return dev_s, dev_t

device_s, device_t = pick_devices()

model_student.to(device_s)
model_teacher.to(device_t)
model_teacher.eval()
for p in model_teacher.parameters():
    p.requires_grad_(False)

proj_s2t = None

optimizer = optim.AdamW(model_student.parameters(), lr=LR)
scaler = GradScaler(enabled=torch.cuda.is_available())
scheduler = get_scheduler(
    name='cosine_with_min_lr',
    optimizer=optimizer,
    num_warmup_steps=int(total_traning_steps * warmup_ratio),
    # num_warmup_steps=1,
    num_training_steps=total_traning_steps,
    scheduler_specific_kwargs={'min_lr': 2e-6}
)
if save_dir:
    os.makedirs(save_dir, exist_ok=True)
warmup_ratio = 0.1
weight_decay = 0.01
max_grad_norm = 1.0
n_items = 0
use_task_loss = True

In [ ]:
def Matry_infonce(a, b, temperature=0.07, nested_dims=[64, 128, 256, 512, 1024]):
    """
    Matryoshka InfoNCE loss - apply contrastive loss on nested dimensions
    
    Args:
        a: [batch_size, full_dim] tensor
        b: [batch_size, full_dim] tensor  
        temperature: temperature for InfoNCE
        nested_dims: list of dimensions to apply loss on
    
    Returns:
        total_loss: sum of losses across all nested dimensions
        all_logits: dict of logits for each dimension
    """
    # Debug: check input shapes
    assert a.dim() == 2, f"Expected a to be 2D [batch, dim], got shape {a.shape}"
    assert b.dim() == 2, f"Expected b to be 2D [batch, dim], got shape {b.shape}"
    assert a.shape == b.shape, f"a and b must have same shape, got {a.shape} vs {b.shape}"
    
    total_loss = 0.0
    all_logits = {}
    
    full_dim = a.size(1)
    
    for dim in nested_dims:
        if dim > full_dim:
            print(f"Warning: skipping dim={dim} as it exceeds full_dim={full_dim}")
            continue
            
        # Slice feature dimension, not batch dimension
        q = a[:, :dim]  # [batch_size, dim]
        k = b[:, :dim]  # [batch_size, dim]
        
        # Normalize
        q = F.normalize(q, dim=-1)
        k = F.normalize(k, dim=-1)
        
        # Compute similarity matrix
        logits = torch.matmul(q, k.T) / temperature  # [batch_size, batch_size]
        
        # Labels: diagonal should match (i-th query matches i-th key)
        labels = torch.arange(q.size(0), device=q.device)
        
        # Cross entropy loss
        loss = F.cross_entropy(logits, labels)
        total_loss += loss
        
        all_logits[f'dim_{dim}'] = logits
    
    return total_loss, all_logits

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import List, Optional, Tuple, Dict, Union, Callable


# ============================================================
# 1. CKALoss Module (Your Implementation)
# ============================================================
class CKALoss(nn.Module):
    """
    CKA (Centered Kernel Alignment) Loss for measuring representation similarity
    
    Computes CKA per-example (using k tokens as samples) then averages over batch.
    """
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def compute_cka_single(self, X, Y):
        """
        Compute CKA for a single example
        
        Args:
            X: [n, p1] - n samples (tokens), p1 features
            Y: [n, p2] - n samples (tokens), p2 features
        
        Returns:
            CKA similarity score (scalar)
        """
        # Convert to float64 for numerical stability
        X = X.to(torch.float64)
        Y = Y.to(torch.float64)
        
        # Center the representations (zero mean across samples)
        X = X - X.mean(0, keepdim=True)
        Y = Y - Y.mean(0, keepdim=True)
        
        # Compute Gram matrices: K = XX^T, L = YY^T
        # Then compute HSIC using Frobenius norm
        # CKA = ||X^T Y||_F^2 / (||X^T X||_F * ||Y^T Y||_F)
        
        XTY = X.t().matmul(Y)  # [p1, p2]
        XTX = X.t().matmul(X)  # [p1, p1]
        YTY = Y.t().matmul(Y)  # [p2, p2]
        
        numerator = torch.norm(XTY, 'fro') ** 2
        denominator = torch.norm(XTX, 'fro') * torch.norm(YTY, 'fro') + self.eps
        
        cka_sim = numerator / denominator
        
        return cka_sim

    def forward(self, SH, TH):
        """
        Args:
            SH: Student hidden states [B, k, d_s] or [k, d_s]
            TH: Teacher hidden states [B, k, d_t] or [k, d_t]
        
        Returns:
            CKA distance (1 - CKA similarity), averaged over batch
        """
        # Check if batched input
        if SH.dim() == 3:  # [B, k, d_s]
            B, k, d_s = SH.shape
            _, _, d_t = TH.shape
            
            # Compute CKA per-example
            cka_sims = []
            for i in range(B):
                cka_sim = self.compute_cka_single(SH[i], TH[i])  # Each is [k, d]
                cka_sims.append(cka_sim)
            
            # Average over batch
            avg_cka_sim = torch.stack(cka_sims).mean()
            
        elif SH.dim() == 2:  # [k, d_s] - single example
            avg_cka_sim = self.compute_cka_single(SH, TH)
            
        else:
            raise ValueError(f"Expected 2D or 3D input, got shape {SH.shape}")
        
        # Return distance (1 - similarity) for minimization
        return 1.0 - avg_cka_sim.float()


# ============================================================
# 2. HorizontalAttentionAlignment Module (FIXED)
# ============================================================
class HorizontalAttentionAlignment(nn.Module):
    """
    Preserves token importance ordering across different dimensions.
    Self-distillation: small dims learn from full dim (1024).
    """
    def __init__(self, d_small: int, d_full: int = 1024, d_att: int = 64):
        """
        Args:
            d_small: Small truncated dimension (e.g., 16, 32, 64, ...)
            d_full: Full dimension (1024 for BERT)
            d_att: Common attention space dimension
        """
        super().__init__()
        self.d_att = d_att
        
        # Projection to common attention space
        self.proj_small = nn.Linear(d_small, d_att)
        self.proj_full = nn.Linear(d_full, d_att)
        
        # Query and Key projections
        self.W_Q = nn.Linear(d_att, d_att)
        self.W_K = nn.Linear(d_att, d_att)
        
    def compute_attention_dist(
        self, 
        hidden: torch.Tensor,  # [B, L, d]
        proj: nn.Linear,
        mask: Optional[torch.Tensor] = None,  # [B, L]
        temperature: float = 1.0
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Compute attention distribution using CLS token as query.
        
        Returns:
            attention_probs: [B, L] - attention distribution over tokens
            scores: [B, L] - raw attention scores
        """
        B, L, _ = hidden.shape
        
        # Project to attention space
        h_proj = proj(hidden)  # [B, L, d_att]
        
        # Get CLS query and all keys
        q_cls = self.W_Q(h_proj[:, 0, :])  # [B, d_att]
        k_all = self.W_K(h_proj)  # [B, L, d_att]
        
        # Compute attention scores
        scores = torch.matmul(q_cls.unsqueeze(1), k_all.transpose(1, 2))  # [B, 1, L]
        scores = scores.squeeze(1) / math.sqrt(self.d_att)  # [B, L]
        
        # Apply mask (set padded positions to -inf)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # Softmax with temperature
        attn_probs = F.softmax(scores / temperature, dim=-1)  # [B, L]
        
        return attn_probs, scores
    
    def forward(
        self,
        h_small: torch.Tensor,  # [B, L, d_small]
        h_full: torch.Tensor,   # [B, L, d_full=1024]
        mask: Optional[torch.Tensor] = None,  # [B, L]
        temperature: float = 1.0
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Compute KL divergence between small and full attention distributions.
        
        Returns:
            loss: KL(small || full)
            small_scores: [B, L] - for use in SubmatrixCKALoss (per-dim selection)
            full_scores: [B, L] - for reference
        """
        # Get attention distributions
        small_probs, small_scores = self.compute_attention_dist(
            h_small, self.proj_small, mask, temperature
        )
        full_probs, full_scores = self.compute_attention_dist(
            h_full, self.proj_full, mask, temperature
        )
        
        # KL divergence: KL(small || full)
        # Add epsilon for numerical stability
        
        eps = 1e-8
        small_probs = small_probs + eps
        full_probs = full_probs + eps
        
        kl_loss = F.kl_div(
            small_probs.log(),
            full_probs,
            reduction='batchmean',
            log_target=False
        )

        full_scores = 0
        kl_loss = 0
        
        return kl_loss, small_scores, full_scores


# ============================================================
# 3. SubmatrixCKALoss Module (FIXED)
# ============================================================
class SubmatrixCKALoss(nn.Module):
    """
    Aligns geometric structure of top-k important tokens using CKA.
    Uses per-dim attention scores for token selection (self-distillation).
    """
    def __init__(self, eps: float = 1e-8):
        super().__init__()
        self.cka_loss = CKALoss(eps=eps)
    
    def select_top_k_tokens(
        self,
        hidden: torch.Tensor,  # [B, L, d]
        selection_scores: torch.Tensor,  # [B, L] - per-dim attention scores
        k: int,
        mask: Optional[torch.Tensor] = None  # [B, L]
    ) -> torch.Tensor:
        """
        Select top-k tokens based on per-dim attention scores.
        
        Args:
            hidden: Hidden states [B, L, d]
            selection_scores: Attention scores for THIS dimension [B, L]
            k: Number of tokens to select
            mask: Attention mask [B, L]
        
        Returns:
            selected_hidden: [B, k, d]
        """
        B, L, d = hidden.shape
        
        # Apply mask to scores (set padded to -inf)
        if mask is not None:
            scores_masked = selection_scores.masked_fill(mask == 0, float('-inf'))
        else:
            scores_masked = selection_scores
        
        # Get top-k indices
        _, top_k_indices = torch.topk(scores_masked, k=min(k, L), dim=1)  # [B, k]
        
        # Gather hidden states
        # Expand indices to match hidden dimension
        top_k_indices_expanded = top_k_indices.unsqueeze(-1).expand(-1, -1, d)  # [B, k, d]
        selected_hidden = torch.gather(hidden, 1, top_k_indices_expanded)  # [B, k, d]
        
        return selected_hidden
    
    def forward(
        self,
        h_small: torch.Tensor,  # [B, L, d_small]
        h_full: torch.Tensor,   # [B, L, d_full=1024]
        small_scores: torch.Tensor,  # [B, L] - per-dim attention scores for small
        k: int,
        mask: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """
        Compute CKA loss on top-k token submatrices.
        Uses per-dim attention scores for selection.
        
        Returns:
            loss: 1 - CKA(small_sub, full_sub)
        """
        # Select top-k tokens using PER-DIM scores
        small_sub = self.select_top_k_tokens(h_small, small_scores, k, mask)  # [B, k, d_small]
        full_sub = self.select_top_k_tokens(h_full, small_scores, k, mask)  # [B, k, d_full]
        
        # Use your CKA loss implementation
        loss = self.cka_loss(small_sub, full_sub)
        
        return loss


# ============================================================
# 4. PipelineInfoNCELoss Module
# ============================================================
class PipelineInfoNCELoss(nn.Module):
    """
    Vertical information chaining across depth using InfoNCE.
    Maps shallow/narrow representations to deeper/wider ones using CLS token.
    Self-distillation: no external teacher model.
    """
    def __init__(self, d_src: int, d_tgt: int, d_hidden: int = 256):
        """
        Args:
            d_src: Source (shallow) dimension
            d_tgt: Target (deep) dimension
            d_hidden: Hidden dimension for MLP
        """
        super().__init__()
        
        # Non-linear projector φ: d_src -> d_tgt
        self.phi = nn.Sequential(
            nn.Linear(d_src, d_hidden),
            nn.ReLU(),
            nn.Linear(d_hidden, d_tgt)
        )
        
    def forward(
        self,
        src_hidden: torch.Tensor,  # [B, L, d_src]
        tgt_hidden: torch.Tensor,  # [B, L, d_tgt]
        mask: Optional[torch.Tensor] = None,  # [B, L] - not used, kept for compatibility
        temperature: float = 0.07
    ) -> torch.Tensor:
        """
        Compute InfoNCE loss between source and target CLS representations.
        
        Returns:
            loss: InfoNCE contrastive loss
        """
        # Extract CLS token (first token) representations
        u_src = src_hidden[:, 0, :]  # [B, d_src]
        v_tgt = tgt_hidden[:, 0, :]  # [B, d_tgt]
        
        # Project source and stop gradient on target
        u_proj = self.phi(u_src)  # [B, d_tgt]
        v_tgt = v_tgt.detach()  # Stop gradient on target
        
        # Normalize for cosine similarity
        u_proj = F.normalize(u_proj, dim=-1)
        v_tgt = F.normalize(v_tgt, dim=-1)
        
        # Compute logits: [B, B] similarity matrix
        logits = torch.matmul(u_proj, v_tgt.T) / temperature
        
        # Labels: diagonal indices (positive pairs)
        labels = torch.arange(logits.size(0), device=logits.device)
        
        # InfoNCE loss (cross entropy)
        loss = F.cross_entropy(logits, labels)
        
        return loss


# ============================================================
# 5. TotalAlignmentLoss Module (FIXED - Main Integration)
# ============================================================
class TotalAlignmentLoss(nn.Module):
    """
    Integrates all three alignment losses for SELF-DISTILLATION:
    L_total = α * L_att + β * L_CKA + γ * L_chain
    
    No external teacher model - uses full dimension (1024) as internal teacher.
    
    FIXES:
    1. Uses per-dim attention scores (small_scores) for token selection
    2. Supports per-dim k_i (nested k) via k_map
    """
    def __init__(
        self,
        d_full: int = 1024,  # Full hidden dimension (BERT base)
        matryoshka_dims: List[int] = [1024, 512, 256, 128, 64, 32, 16],
        align_layers: List[int] = [4, 8],  # Layers to align
        pipeline_pairs: List[Tuple[int, ...]] = None,  # Sequential pairs
        alpha: float = 1.0,  # Weight for attention loss
        beta: float = 1.0,   # Weight for CKA loss
        gamma: float = 1.0,  # Weight for InfoNCE loss
        k_map: Optional[Union[Dict[int, int], Callable[[int], int]]] = None,  # Per-dim k_i
        base_k: int = 32,    # Base k for computing default k_map
        temperature: float = 1.0,
        d_att: int = 64
    ):
        super().__init__()
        
        self.d_full = d_full
        self.matryoshka_dims = sorted([d for d in matryoshka_dims if d <= d_full], reverse=True)
        self.align_layers = align_layers
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.temperature = temperature
        
        # Setup k_map: dim -> k_i (nested, monotone increasing)
        if k_map is None:
            # Default: k_i proportional to dim ratio, with minimum 8
            self.k_map = lambda d: max(8, int((d / d_full) * base_k))
        elif callable(k_map):
            self.k_map = k_map
        elif isinstance(k_map, dict):
            self.k_map = lambda d: k_map.get(d, max(8, int((d / d_full) * base_k)))
        else:
            raise ValueError("k_map must be None, callable, or dict")
        
        # Parse pipeline_pairs into consecutive pairs
        if pipeline_pairs is None:
            pipeline_pairs = [(3, 16, 5, 32, 7, 64, 8, 128, 9, 256, 10, 512, 11, 1024)]
        
        self.parsed_pipeline_pairs = []
        for pipe_config in pipeline_pairs:
            if len(pipe_config) % 2 != 0:
                raise ValueError(f"Pipeline config must have even length: {pipe_config}")
            
            # Extract (layer, dim) pairs
            stages = [(pipe_config[i], pipe_config[i+1]) for i in range(0, len(pipe_config), 2)]
            
            # Create consecutive pairs
            for i in range(len(stages) - 1):
                layer_i, dim_i = stages[i]
                layer_j, dim_j = stages[i + 1]
                self.parsed_pipeline_pairs.append((layer_i, dim_i, layer_j, dim_j))
        
        # Create attention alignment modules for each layer and dimension
        self.attn_modules = nn.ModuleDict()
        for layer_idx in align_layers:
            for dim in self.matryoshka_dims:
                if dim >= d_full:  # Skip full dimension (it's the teacher)
                    continue
                key = f"layer_{layer_idx}_dim_{dim}"
                self.attn_modules[key] = HorizontalAttentionAlignment(
                    d_small=dim,
                    d_full=d_full,
                    d_att=d_att
                )
        
        # Single CKA module (dimension-agnostic)
        self.cka_module = SubmatrixCKALoss()
        
        # Create InfoNCE modules for each pipeline pair
        self.infonce_modules = nn.ModuleDict()
        for layer_i, dim_i, layer_j, dim_j in self.parsed_pipeline_pairs:
            key = f"pipe_{layer_i}_{dim_i}_to_{layer_j}_{dim_j}"
            self.infonce_modules[key] = PipelineInfoNCELoss(
                d_src=dim_i,
                d_tgt=dim_j,
                d_hidden=max(dim_i, dim_j) // 2
            )
    
    def forward(
        self,
        hidden_states: List[torch.Tensor],  # List of [B, L, d_full] for each layer
        mask: Optional[torch.Tensor] = None  # [B, L]
    ) -> Dict[str, torch.Tensor]:
        """
        Compute total alignment loss for SELF-DISTILLATION.
        
        Args:
            hidden_states: List of hidden states from student model (all are d_full=1024)
            mask: Attention mask
            
        Returns:
            Dictionary containing total loss and individual components
        """
        total_att_loss = 0.0
        total_cka_loss = 0.0
        total_chain_loss = 0.0
        
        att_count = 0
        cka_count = 0
        
        # ========== Horizontal Alignment (FIXED) ==========
        for layer_idx in self.align_layers:
            full_layer = hidden_states[layer_idx]  # [B, L, d_full=1024]
            
            for dim in self.matryoshka_dims:
                if dim >= self.d_full:  # Skip full dimension
                    continue
                
                # Truncate to small dimension
                small_layer = full_layer[..., :dim]  # [B, L, dim]
                
                # Get per-dim k_i (nested, monotone increasing with dim)
                k_i = self.k_map(dim)
                
                # Get attention alignment module
                key = f"layer_{layer_idx}_dim_{dim}"
                attn_module = self.attn_modules[key]
                
                # Compute attention loss and get PER-DIM scores
                att_loss, small_scores, full_scores = attn_module(
                    h_small=small_layer,
                    h_full=full_layer,
                    mask=mask,
                    temperature=self.temperature
                )
                
                total_att_loss += att_loss
                att_count += 1
                
                # Compute CKA loss using PER-DIM guided selection with k_i
                cka_loss = self.cka_module(
                    h_small=small_layer,
                    h_full=full_layer,
                    small_scores=small_scores,  # Use per-dim scores!
                    k=k_i,  # Use per-dim k_i!
                    mask=mask
                )
                
                total_cka_loss += cka_loss
                cka_count += 1
        
        # Average horizontal losses
        if att_count > 0:
            total_att_loss = total_att_loss / att_count
        if cka_count > 0:
            total_cka_loss = total_cka_loss / cka_count
        
        # ========== Vertical Chaining ==========
        for layer_i, dim_i, layer_j, dim_j in self.parsed_pipeline_pairs:
            # Extract and truncate hidden states
            src_hidden = hidden_states[layer_i][..., :dim_i]  # [B, L, dim_i]
            tgt_hidden = hidden_states[layer_j][..., :dim_j]  # [B, L, dim_j]
            
            key = f"pipe_{layer_i}_{dim_i}_to_{layer_j}_{dim_j}"
            infonce_module = self.infonce_modules[key]
            
            chain_loss = infonce_module(
                src_hidden=src_hidden,
                tgt_hidden=tgt_hidden,
                mask=mask,
                temperature=0.07
            )
            
            total_chain_loss += chain_loss
        
        
        total_chain_loss = total_chain_loss / len(self.parsed_pipeline_pairs)
        
        # ========== Total Loss ==========
        total_loss = (
            self.alpha * total_att_loss +
            self.beta * total_cka_loss +
            self.gamma * total_chain_loss
        )
        
        return {
            'total_loss': total_loss,
            'att_loss': total_att_loss,
            'cka_loss': total_cka_loss,
            'chain_loss': total_chain_loss
        }

In [ ]:
# 4. Matryoshka Alignment Loss Module Setup
# ============================================================
print("\nInitializing Matryoshka Alignment Loss Module...")

# Param for bert base

# matryoshka_dims = [1024, 512, 256, 128, 64, 32, 16]
# align_layers = [8, 10]  # For 12-layer BERT: layers 4, 8, 11
# pipeline_pairs = [(3, 16, 7, 128, 11, 1024)]

# Param for tinybert 6L
# matryoshka_dims = [1024, 512, 256, 128, 64, 32, 16]
# align_layers = [2, 4]  # For 12-layer BERT: layers 4, 8, 11
# pipeline_pairs = [(1, 16, 3, 128, 5, 1024)]

matryoshka_dims = [1024, 512, 256, 128, 64, 32, 16]
align_layers = [2, 8,14,20, 23]  
pipeline_pairs = [(2, 16, 8, 64,14,128, 20, 512, 23, 1024)]


base_k = 64  # Max k for full dimension

# Option 2: Custom k_map (uncomment to use)
# k_map = {
#     16: 8,
#     32: 12,
#     64: 16,
#     128: 20,
#     256: 24,
#     512: 28,
#     1024: 32
# }

#Option 3: Custom function (uncomment to use)
k_map = lambda d: min(64, max(8, d // 20))

# Initialize alignment loss module
alignment_loss_module = TotalAlignmentLoss(
    d_full=1024,
    matryoshka_dims=matryoshka_dims,
    align_layers=align_layers,
    pipeline_pairs=pipeline_pairs,
    alpha=0.4,   # Weight for attention alignment loss
    beta=0.4,    # Weight for CKA loss
    gamma=0.2,   # Weight for InfoNCE chain loss (typically smaller)
    k_map=None,  # Use default proportional mapping
    base_k=base_k,
    temperature=1.0,
    d_att=64
).to(device_s)

In [ ]:
# ========== Training Loop với Token-Level CKA ==========
from torch.cuda.amp import autocast, GradScaler
import torch.optim as optim

def info_nce(q, k, temperature=0.07):
    """InfoNCE contrastive loss"""
    q = F.normalize(q, dim=-1)
    k = F.normalize(k, dim=-1)
    logits = torch.matmul(q, k.T) / temperature
    labels = torch.arange(q.size(0), device=q.device)
    loss = F.cross_entropy(logits, labels)
    return loss, logits

# Collect all parameters
all_params = list(model_student.parameters()) + list(alignment_loss_module.parameters())

# Create optimizer with parameter groups
optimizer = torch.optim.AdamW([
    {'params': model_student.parameters(), 'lr': LR},
    {'params': alignment_loss_module.parameters(), 'lr': LR * 2}  # Higher LR for alignment module
], weight_decay=weight_decay)

# ========== Hyperparameters ==========
w_task = 0.5       # Task loss weight
w_loss = 0.5       # final loss weight

matryoshka_dims = [1024, 512, 256, 128, 64, 32, 16]  # Descending order!
n_dims_per_step = -1  # -1 = use all dims, or set to 3-4 for efficiency

# ========== Training Loop ==========
for epoch in range(epochs):
    model_student.train()
    alignment_loss_module.train()
    total_loss, n_items = 0.0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for batch in pbar:
        batch_s = {}
        for k, v in batch.items():
            if not torch.is_tensor(v):
                continue
            if k.endswith("_stu"):
                batch_s[k] = v.to(device_s, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=torch.cuda.is_available()):
            # ========== STUDENT Forward Pass ==========
            s_out1 = model_student(
                input_ids=batch_s["input_ids1_stu"],
                attention_mask=batch_s["attention_mask1_stu"],
                output_hidden_states=True,
                return_dict=True
            )
            s_out2 = model_student(
                input_ids=batch_s["input_ids2_stu"],
                attention_mask=batch_s["attention_mask2_stu"],
                output_hidden_states=True,
                return_dict=True
            )
            
            # Get token-level hidden states [B, L, hidden_dim]
            S_last1 = s_out1.hidden_states[-1]  # [B, L1, 1024]
            S_last2 = s_out2.hidden_states[-1]  # [B, L2, 1024]
            
            S_cls1  = S_last1[:, 0, :]               # [B, d_s]
            S_cls2  = S_last2[:, 0, :]

            # ========== (A) Task Loss: SimCSE InfoNCE ==========
            #loss_task, _ = info_nce(S_cls1, S_cls2, temperature=0.07)
            loss_task = 0

            # ========== (B) Matryoshka Alignment Loss ==========
            # Compute alignment for first pair (premise)
            loss_dict1 = alignment_loss_module(
                hidden_states=list(s_out1.hidden_states),  # Just student hidden states
                mask=batch_s["attention_mask1_stu"]
            )

            # Compute alignment for second pair (hypothesis)
            loss_dict2 = alignment_loss_module(
                hidden_states=list(s_out2.hidden_states),  # Just student hidden states
                mask=batch_s["attention_mask2_stu"]
            )

            nested_dims = [16, 32, 64, 128, 256, 512, 1024]  # Adjust based on your model's hidden size
            cka_loss, _ = Matry_infonce(S_cls1, S_cls2, temperature=0.07, nested_dims=nested_dims)

            # Average alignment losses from both pairs
            alignment_total = (loss_dict1['total_loss'] + loss_dict2['total_loss']) / 2.0
            alignment_att = (loss_dict1['att_loss'] + loss_dict2['att_loss']) / 2.0
            alignment_cka = (loss_dict1['cka_loss'] + loss_dict2['cka_loss']) / 2.0
            alignment_chain = (loss_dict1['chain_loss'] + loss_dict2['chain_loss']) / 2.0


            # ========== Combined Loss ==========
            #loss = w_task * loss_task + w_loss * alignment_total

            loss = 0.4*alignment_total + 0.6*cka_loss
            loss = loss.float()

        # Backward pass
        scaler.scale(loss).backward()
        
        # Gradient clipping (optional but recommended)
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model_student.parameters(), max_norm=1.0)
        torch.nn.utils.clip_grad_norm_(alignment_loss_module.parameters(), max_norm=1.0)
        
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        # ----- Logging -----
        bs = batch_s["input_ids1_stu"].size(0)
        total_loss += loss.item() * bs
        n_items += bs

        mem_info = {}
        for dev_id in range(torch.cuda.device_count()):
            mem_alloc = torch.cuda.memory_allocated(dev_id) / 1024**2
            mem_reserved = torch.cuda.memory_reserved(dev_id) / 1024**2
            mem_info[f"gpu{dev_id}"] = f"{mem_alloc:.0f}/{mem_reserved:.0f}MB"

        avg_loss = total_loss / max(1, n_items)
        pbar.set_postfix({
            "avg_loss": f"{avg_loss:.4f}",
            #"task": f"{loss_task.item():.4f}",
            "align": f"{alignment_total.item():.4f}",
            #"att": f"{alignment_att.item():.4f}",
            "cka": f"{alignment_cka.item():.4f}",
            "chain": f"{alignment_chain.item():.4f}",
            **mem_info
        })

        # Clean up
        del s_out1, s_out2
        del S_cls1, S_cls2
        del loss, loss_task, alignment_total
        torch.cuda.empty_cache()

    # Evaluation
    print(f"\n=== Epoch {epoch+1} Evaluation ===")
    eval_classification_task(model_student, test_cls_tasks)
    eval_pair_task(model_student, test_pair_tasks)
    eval_sts_task(model_student, test_sts_tasks)
    
   